In [1]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'example' else Path.cwd()))
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (11, 3.5)

from src import Dataset

ds = Dataset()
ds

<Dataset 50종목 기준일 2024-08-13~2026-09-11 522일 (일봉 2023-10-23~2026-09-14)>

In [3]:
daily = ds.table('daily')

print("shape:", daily.shape)

print("\n[Missing values]")
print(daily.isna().sum())

print("\n[Duplicate rows]")
print(daily.duplicated().sum())

print("\n[Unique symbols]")
print(daily["symbol"].nunique())

print("\n[Rows per symbol]")
print(daily.groupby("symbol").size().sort_values())

print("\n[Date range by symbol]")
coverage = daily.groupby("symbol")["date_et"].agg(["min", "max", "count"])
display(coverage)

shape: (36293, 11)

[Missing values]
symbol         0
date_et        0
open           0
high           0
low            0
close          0
volume         0
prev_close    50
ret           50
up             0
known_at       0
dtype: int64

[Duplicate rows]
0

[Unique symbols]
50

[Rows per symbol]
symbol
TMO      719
ADI      726
AMGN     726
AMD      726
AVGO     726
AXP      726
BA       726
AMZN     726
BRK-B    726
CB       726
COP      726
COST     726
CRM      726
CSCO     726
CVS      726
BKNG     726
DELL     726
DHR      726
GE       726
GILD     726
GLW      726
GS       726
IBKR     726
INTC     726
JNJ      726
KO       726
LIN      726
LLY      726
MCD      726
MS       726
MSFT     726
DE       726
NEM      726
NFLX     726
ORCL     726
NOW      726
PLD      726
PM       726
QCOM     726
PFE      726
RTX      726
TJX      726
TMUS     726
UBER     726
UNP      726
V        726
VZ       726
WELL     726
WMT      726
XOM      726
dtype: int64

[Date range by symbol]


,min,max,count
symbol,,,
ADI,2023-10-20,2026-09-14,726
AMD,2023-10-20,2026-09-14,726
AMGN,2023-10-20,2026-09-14,726
AMZN,2023-10-20,2026-09-14,726
AVGO,2023-10-20,2026-09-14,726
AXP,2023-10-20,2026-09-14,726
BA,2023-10-20,2026-09-14,726
BKNG,2023-10-20,2026-09-14,726
BRK-B,2023-10-20,2026-09-14,726


In [4]:
# 1) 종목별 날짜 범위 확인
coverage = daily.groupby("symbol")["date_et"].agg(["min", "max", "count"])

print("TMO coverage")
display(coverage.loc[["TMO"]])

print("\nOther symbols sample")
display(coverage.head())

TMO coverage


,min,max,count
symbol,,,
TMO,2023-10-31,2026-09-14,719



Other symbols sample


,min,max,count
symbol,,,
ADI,2023-10-20,2026-09-14,726
AMD,2023-10-20,2026-09-14,726
AMGN,2023-10-20,2026-09-14,726
AMZN,2023-10-20,2026-09-14,726
AVGO,2023-10-20,2026-09-14,726


In [8]:
# 날짜순 정렬
daily = daily.sort_values(["symbol", "date_et"]).copy()

# 1) 최근 5거래일 누적 수익률
daily["ret_5d"] = (
    daily.groupby("symbol")["close"]
    .pct_change(5)
)

# 2) 최근 5거래일 변동성
daily["vol_5d"] = (
    daily.groupby("symbol")["ret"]
    .rolling(5)
    .std()
    .reset_index(level=0, drop=True)
)

daily[
    ["symbol", "date_et", "ret", "ret_5d", "vol_5d"]
].head(15)

,symbol,date_et,ret,ret_5d,vol_5d
17,ADI,2023-10-20,NaN,NaN,NaN
84,ADI,2023-10-23,-0.015855,NaN,NaN
130,ADI,2023-10-24,0.006469,NaN,NaN
178,ADI,2023-10-25,-0.031286,NaN,NaN
216,ADI,2023-10-26,0.006384,NaN,NaN
256,ADI,2023-10-27,-0.001306,-0.035613,0.016287
339,ADI,2023-10-30,-0.029331,-0.048819,0.018986
383,ADI,2023-10-31,0.009623,-0.045838,0.019701
408,ADI,2023-11-01,0.011946,-0.003255,0.016857
462,ADI,2023-11-02,0.031837,0.021954,0.022401


ret: 당일 수익률
ret_5d: 5거래일 전 대비 현재까지 누적 수익률
vol_5d: 최근 5거래일 수익률의 표준편차
처음 5개 정도가 NaN인 건 과거 5일 데이터가 없어서 정상

In [9]:
# 다음 거래일 수익률
daily["next_ret"] = (
    daily.groupby("symbol")["ret"]
    .shift(-1)
)

# 다음 거래일 5-class label
def make_label(r):
    if pd.isna(r):
        return np.nan
    elif r < -0.025:
        return 0
    elif r < -0.006:
        return 1
    elif r <= 0.006:
        return 2
    elif r <= 0.025:
        return 3
    else:
        return 4

daily["next_label"] = daily["next_ret"].apply(make_label)

daily[
    ["symbol", "date_et", "ret", "ret_5d", "vol_5d",
     "next_ret", "next_label"]
].head(10)

,symbol,date_et,ret,ret_5d,vol_5d,next_ret,next_label
17,ADI,2023-10-20,NaN,NaN,NaN,-0.015855,1.0
84,ADI,2023-10-23,-0.015855,NaN,NaN,0.006469,3.0
130,ADI,2023-10-24,0.006469,NaN,NaN,-0.031286,0.0
178,ADI,2023-10-25,-0.031286,NaN,NaN,0.006384,3.0
216,ADI,2023-10-26,0.006384,NaN,NaN,-0.001306,2.0
256,ADI,2023-10-27,-0.001306,-0.035613,0.016287,-0.029331,0.0
339,ADI,2023-10-30,-0.029331,-0.048819,0.018986,0.009623,3.0
383,ADI,2023-10-31,0.009623,-0.045838,0.019701,0.011946,3.0
408,ADI,2023-11-01,0.011946,-0.003255,0.016857,0.031837,4.0
462,ADI,2023-11-02,0.031837,0.021954,0.022401,0.021482,3.0


In [10]:
feature_cols = ["ret", "ret_5d", "vol_5d"]

analysis_df = daily.dropna(
    subset=feature_cols + ["next_ret"]
).copy()

print("[Spearman correlation with next-day return]")

for col in feature_cols:
    corr = analysis_df[col].corr(
        analysis_df["next_ret"],
        method="spearman"
    )
    print(f"{col:10s}: {corr:.4f}")

[Spearman correlation with next-day return]
ret       : -0.0086
ret_5d    : -0.0177
vol_5d    : 0.0157


셋 다 거의 0에 가까움 -> 단독으로는 다음날 수익률과 monotonic relationship이 거의 없음

In [11]:
def quintile_analysis(df, feature):
    temp = df[[feature, "next_ret", "next_label"]].dropna().copy()
    temp["q"] = pd.qcut(temp[feature], 5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"])

    result = temp.groupby("q", observed=True).agg(
        count=("next_ret", "size"),
        mean_next_ret=("next_ret", "mean"),
        up_rate=("next_ret", lambda x: (x > 0).mean()),
        big_move_rate=("next_label", lambda x: x.isin([0, 4]).mean())
    )

    return result

for feature in ["ret", "ret_5d", "vol_5d"]:
    print(f"\n===== {feature} =====")
    display(quintile_analysis(daily, feature))


===== ret =====


,count,mean_next_ret,up_rate,big_move_rate
q,,,,
Q1,7239,0.001683,0.536400,0.192154
Q2,7238,0.000794,0.519757,0.108455
Q3,7239,0.000989,0.531980,0.100843
Q4,7238,0.000730,0.527217,0.109284
Q5,7239,0.001387,0.530184,0.188838



===== ret_5d =====


,count,mean_next_ret,up_rate,big_move_rate
q,,,,
Q1,7199,0.001685,0.538825,0.200445
Q2,7198,0.001221,0.534037,0.111837
Q3,7199,0.000770,0.523545,0.107237
Q4,7198,0.000921,0.533343,0.103779
Q5,7199,0.001088,0.518822,0.175302



===== vol_5d =====


,count,mean_next_ret,up_rate,big_move_rate
q,,,,
Q1,7199,0.000637,0.526323,0.061814
Q2,7198,0.000987,0.527924,0.082245
Q3,7199,0.000742,0.528823,0.113488
Q4,7198,0.001414,0.538761,0.157544
Q5,7199,0.001905,0.526740,0.283512


ret : 전날 return이 아주 크거나 아주 작았던 날 (Q1,Q5) 다음 날에도 급등/급락이 발생할 확률이 높아지는 U자형 패턴
-> raw ret 보다는 abs(ret)가 유용할 가능성..?
ret_5d : ret와 비슷
vol_5d : 가장 깔끔, 거의 완벽하게 증가( 최근 5일 volatility가 높은 종목일술고 다음 거래일에 급등/급락이 발생할 가능성이 높음)
-> 내일 얘가 평범하게 움직일 종목인지 크게 움직일 종목인지 판단하는데 도움 가능
-> 왜 correlation 0 가까움? -> 큰 움직임이 부호 달라서 서로 상쇄 -> 수익률의 절댓값을 보자


In [12]:
# 큰 움직임의 크기 자체를 보기 위한 절댓값 feature/target
daily["abs_ret"] = daily["ret"].abs()
daily["abs_ret_5d"] = daily["ret_5d"].abs()
daily["abs_next_ret"] = daily["next_ret"].abs()

features = ["abs_ret", "abs_ret_5d", "vol_5d"]

print("[Spearman correlation with absolute next-day return]")

for col in features:
    temp = daily[[col, "abs_next_ret"]].dropna()

    corr = temp[col].corr(
        temp["abs_next_ret"],
        method="spearman"
    )

    print(f"{col:12s}: {corr:.4f}")

[Spearman correlation with absolute next-day return]
abs_ret     : 0.1433
abs_ret_5d  : 0.1351
vol_5d      : 0.2368


ret과 ret_5d는 다음날 방향 예측에는 거의 도움 X -> 절댓값 기준으로는 다음날 움직임의 크기와 양의 관계가 있음

특히 vol_5d는 abs(next_ret)과 spearman 0.2368을 보임 -> 다음날 급등락 가능성을 설명하는 feature 후보

In [13]:
news = ds.table("news").copy()

print(news.shape)
print(news.columns.tolist())
news.head()

(3203409, 9)
['symbols', 'source', 'url', 'title', 'tone', 'positive', 'negative', 'known_at', 'text']


,symbols,source,url,title,tone,positive,negative,known_at,text
0,"MS,WFC",nbcnewyork.com,https://www.nbcnewyork.com/news/business/money...,Stock futures are little changed ahead of key ...,0.299401,2.095808,1.796407,2024-08-13 20:00:00,NaN
1,AVGO,investorplace.com,https://investorplace.com/market360/2024/08/te...,Tech Earnings Alert! Broadcom Stock May Be the...,2.355072,3.442029,1.086957,2024-08-13 20:00:00,NaN
2,NFLX,cairnspost.com.au,https://www.cairnspost.com.au/lifestyle/when-i...,Where Emily in Paris goes to in Rome in season 4,1.234568,3.254770,2.020202,2024-08-13 20:00:00,NaN
3,"GS,C",fox5ny.com,https://www.fox5ny.com/news/fraud-trial-george...,Fraud trial of George Santos to begin next mon...,-5.429293,1.136364,6.565657,2024-08-13 20:00:00,NaN
4,"MSFT,ORCL",sina.com.hk,https://portal.sina.com.hk/finance/finance-glo...,"For the Third Year in a Row, Anteriad Makes th...",2.387775,3.724928,1.337154,2024-08-13 20:00:00,NaN


In [14]:
news_exploded = news.explode("symbols").rename(columns={"symbols": "symbol"}).copy()

news_exploded[["symbol", "known_at", "tone", "positive", "negative"]].head()

,symbol,known_at,tone,positive,negative
0,"MS,WFC",2024-08-13 20:00:00,0.299401,2.095808,1.796407
1,AVGO,2024-08-13 20:00:00,2.355072,3.442029,1.086957
2,NFLX,2024-08-13 20:00:00,1.234568,3.254770,2.020202
3,"GS,C",2024-08-13 20:00:00,-5.429293,1.136364,6.565657
4,"MSFT,ORCL",2024-08-13 20:00:00,2.387775,3.724928,1.337154


In [20]:
news = ds.table("news").copy()

# "AAPL,ABBV,COST" 같은 문자열을 list로 변환한 뒤 explode
news["symbols"] = news["symbols"].str.split(",")

news_exploded = (
    news
    .explode("symbols")
    .rename(columns={"symbols": "symbol"})
    .copy()
)

# 혹시 공백이 있으면 제거
news_exploded["symbol"] = news_exploded["symbol"].str.strip()

news_exploded[
    ["symbol", "known_at", "tone", "positive", "negative"]
].head(10)

,symbol,known_at,tone,positive,negative
0,MS,2024-08-13 20:00:00,0.299401,2.095808,1.796407
0,WFC,2024-08-13 20:00:00,0.299401,2.095808,1.796407
1,AVGO,2024-08-13 20:00:00,2.355072,3.442029,1.086957
2,NFLX,2024-08-13 20:00:00,1.234568,3.254770,2.020202
3,GS,2024-08-13 20:00:00,-5.429293,1.136364,6.565657
3,C,2024-08-13 20:00:00,-5.429293,1.136364,6.565657
4,MSFT,2024-08-13 20:00:00,2.387775,3.724928,1.337154
4,ORCL,2024-08-13 20:00:00,2.387775,3.724928,1.337154
5,WMT,2024-08-13 20:00:00,5.000000,6.186441,1.186441
5,DE,2024-08-13 20:00:00,5.000000,6.186441,1.186441


In [21]:
news_exploded["date_et"] = pd.to_datetime(
    news_exploded["known_at"]
).dt.date

news_daily = (
    news_exploded
    .groupby(["symbol", "date_et"])
    .agg(
        news_count=("symbol", "size"),
        mean_tone=("tone", "mean"),
        mean_positive=("positive", "mean"),
        mean_negative=("negative", "mean")
    )
    .reset_index()
)

news_daily.head(10)

,symbol,date_et,news_count,mean_tone,mean_positive,mean_negative
0,AAPL,2024-08-13,5,0.478629,3.363160,2.884531
1,AAPL,2024-08-14,94,1.048098,3.307290,2.259192
2,AAPL,2024-08-15,85,1.567217,3.618204,2.050987
3,AAPL,2024-08-16,63,1.690260,3.816109,2.125849
4,AAPL,2024-08-17,39,2.129401,3.726577,1.597176
5,AAPL,2024-08-18,28,1.232759,3.236850,2.004091
6,AAPL,2024-08-19,67,0.778192,2.926159,2.147968
7,AAPL,2024-08-20,77,1.412216,3.487913,2.075697
8,AAPL,2024-08-21,74,1.106765,3.377196,2.270430
9,AAPL,2024-08-22,91,1.250283,3.376861,2.126577


In [22]:
symbol_median_count = (
    news_daily
    .groupby("symbol")["news_count"]
    .median()
    .rename("median_news_count")
)

news_daily = news_daily.merge(
    symbol_median_count,
    on="symbol",
    how="left"
)

news_daily["news_count_ratio"] = (
    news_daily["news_count"]
    / news_daily["median_news_count"]
)

news_daily[
    ["symbol", "date_et", "news_count",
     "median_news_count", "news_count_ratio",
     "mean_tone"]
].head(10)

,symbol,date_et,news_count,median_news_count,news_count_ratio,mean_tone
0,AAPL,2024-08-13,5,68.0,0.073529,0.478629
1,AAPL,2024-08-14,94,68.0,1.382353,1.048098
2,AAPL,2024-08-15,85,68.0,1.250000,1.567217
3,AAPL,2024-08-16,63,68.0,0.926471,1.690260
4,AAPL,2024-08-17,39,68.0,0.573529,2.129401
5,AAPL,2024-08-18,28,68.0,0.411765,1.232759
6,AAPL,2024-08-19,67,68.0,0.985294,0.778192
7,AAPL,2024-08-20,77,68.0,1.132353,1.412216
8,AAPL,2024-08-21,74,68.0,1.088235,1.106765
9,AAPL,2024-08-22,91,68.0,1.338235,1.250283


news_count ratio > 1 이면 평소보다 뉴스가 많이 나온 날 -> raw count보다 종목별 평소 기사량 대비 ratio를 쓰자

But news_daily를 그대로 target과 merge하면 안됨 -> 하루 전체 뉴스를 묶었기 떄문에 예측일 09:30 이후에 나온 뉴스까지 섞이는 leakage 생길 수 있음
->  Dataset/Day API 이용

In [23]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'example' else Path.cwd()))
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (11, 3.5)

from src import Dataset

ds = Dataset()
ds

<Dataset 50종목 기준일 2024-08-13~2026-09-11 522일 (일봉 2023-10-23~2026-09-14)>

In [24]:
day = ds.day("2026-02-20")

print("base date :", day.date)
print("target    :", day.target)
print("cutoff    :", day.cutoff)

sample_news = day.news(["AMD"], days=1)

print("rows:", len(sample_news))
display(sample_news.head())

print("latest known_at:", sample_news["known_at"].max())

base date : 2026-02-20 00:00:00
target    : 2026-02-23 00:00:00
cutoff    : 2026-02-23 09:30:00
rows: 56


,known_at,symbols,source,url,title,tone,positive,negative
0,2026-02-20 00:15:00,"AMD,WFC",dailypolitical.com,https://www.dailypolitical.com/2026/02/19/adva...,Advanced Micro Devices (NASDAQ:AMD) Trading Do...,3.359859,3.801945,0.442087
1,2026-02-20 00:45:00,"AMD,INTC,GS",yahoo.com,https://finance.yahoo.com/news/jim-cramer-beli...,Jim Cramer Believes Advanced Micro Devices (AM...,0.597015,3.283582,2.686567
2,2026-02-20 03:30:00,"NVDA,AVGO,AMD,ORCL,NFLX",yahoo.com,https://finance.yahoo.com/news/amd-vs-broadcom...,AMD vs. Broadcom: Which One Will Dominate the ...,-0.641026,2.777778,3.418803
3,2026-02-20 04:15:00,AMD,newegg.com,https://www.newegg.com/amd-ryzen-9-9000-series...,AMD Ryzen 9 9950X3D - Ryzen 9 9000 Series Gran...,-0.519481,3.636364,4.155844
4,2026-02-20 05:00:00,"NVDA,AMD",insidermonkey.com,https://www.insidermonkey.com/blog/benchmark-s...,Benchmark Sees Advanced Micro Devices (AMD) Ge...,2.032520,5.691057,3.658537


latest known_at: 2026-02-23 09:15:00


In [25]:
sample_news = day.news(["AMD"], days=1).copy()

news_count = len(sample_news)
mean_tone = sample_news["tone"].mean()

print("news_count :", news_count)
print("mean_tone  :", mean_tone)

news_count : 56
mean_tone  : 1.6427073374045382


해석: 예측 시점까지 AMD 뉴스가 56개 있었고 전체적으로는 약간 긍정적인 분위기

- news_count → 뉴스가 얼마나 많이 나왔는지
- mean_tone → 뉴스 분위기가 평균적으로 긍정/부정 중 어느 쪽인지
- news_count_ratio → 평소보다 뉴스가 얼마나 많이 나온 날인지
tone은 데이터셋에 이미 들어있는 뉴스 감성 점수 -> 보통 양수면 더 긍정적, 음수면 더 부정적인 쪽으로 해석

In [26]:
# 현재 예측 시점까지 최근 20거래일 AMD 뉴스
hist_news = day.news(["AMD"], days=20).copy()

hist_news["date_et"] = pd.to_datetime(
    hist_news["known_at"]
).dt.date

# 날짜별 뉴스 개수
daily_counts = (
    hist_news
    .groupby("date_et")
    .size()
)

# 최근 20일의 평소 뉴스량
median_20d = daily_counts.median()

# 현재 cutoff까지 뉴스 개수
current_news_count = len(day.news(["AMD"], days=1))

# normalized news count
news_count_ratio = (
    current_news_count / median_20d
    if median_20d > 0 else np.nan
)

print("current_news_count :", current_news_count)
print("20-day median      :", median_20d)
print("news_count_ratio   :", news_count_ratio)

current_news_count : 56
20-day median      : 31.0
news_count_ratio   : 1.8064516129032258


-> normalized news count 만들기
* 전체 기간 median 쓰지 않고 각 예측일 기준으로 과거 뉴스량만 이용해서 normalize
- ratio = 1 -> 평소 수준
- ratio = 2 -> 평소보다 약 2배 많은 뉴스
- ratio = 0.5 -> 평소의 절반 정도

In [27]:
def get_news_features(day, symbol, history_days=20):
    # 최근 1일 뉴스
    current_news = day.news([symbol], days=1).copy()

    # 최근 history_days일 뉴스
    hist_news = day.news([symbol], days=history_days).copy()

    # 1) 현재 뉴스 개수
    news_count = len(current_news)

    # 2) 현재 뉴스 평균 tone
    mean_tone = (
        current_news["tone"].mean()
        if len(current_news) > 0
        else np.nan
    )

    # 3) 최근 기간 기준 normalized news count
    if len(hist_news) > 0:
        hist_news["date_et"] = pd.to_datetime(
            hist_news["known_at"]
        ).dt.date

        daily_counts = (
            hist_news
            .groupby("date_et")
            .size()
        )

        median_count = daily_counts.median()

        news_count_ratio = (
            news_count / median_count
            if median_count > 0
            else np.nan
        )
    else:
        news_count_ratio = np.nan

    return {
        "news_count": news_count,
        "news_count_ratio": news_count_ratio,
        "mean_tone": mean_tone
    }

In [28]:
get_news_features(day, "AMD")

{'news_count': 56,
 'news_count_ratio': np.float64(1.8064516129032258),
 'mean_tone': np.float64(1.6427073374045382)}

먼저 몇개만 돌려봄

In [31]:
test_rows = []

# 날짜 3개만 테스트
test_days = [
    ds.day("2026-02-18"),
    ds.day("2026-02-19"),
    ds.day("2026-02-20")
]

# 종목 3개만 테스트
test_symbols = ["AMD", "AAPL", "NVDA"]

for d in test_days:
    for symbol in test_symbols:
        features = get_news_features(d, symbol)

        test_rows.append({
            "date_et": d.date,
            "target": d.target,
            "symbol": symbol,
            **features
        })

test_news_features = pd.DataFrame(test_rows)

test_news_features

,date_et,target,symbol,news_count,news_count_ratio,mean_tone
0,2026-02-18,2026-02-19,AMD,48,1.500000,1.712080
1,2026-02-18,2026-02-19,AAPL,69,1.086614,0.831210
2,2026-02-18,2026-02-19,NVDA,692,2.302829,0.387874
3,2026-02-19,2026-02-20,AMD,31,0.968750,0.703595
4,2026-02-19,2026-02-20,AAPL,89,1.358779,1.180972
5,2026-02-19,2026-02-20,NVDA,569,1.859477,0.477697
6,2026-02-20,2026-02-23,AMD,56,1.806452,1.642707
7,2026-02-20,2026-02-23,AAPL,130,2.166667,1.144126
8,2026-02-20,2026-02-23,NVDA,728,2.523397,0.562315


-> news_count_ration 를 보면 각 종목의 평소 수준 대비 얼마나 이례적인지 비교 가능

+ 수정: 20일 meadian이 뉴스가 한 건도 없었던 날짜를 아예 제외하고 계산 -> 수정해야함 !!

In [32]:
def get_news_features(day, symbol, history_days=20):
    # 현재 예측 시점까지 최근 1거래일 뉴스
    current_news = day.news([symbol], days=1).copy()

    # 최근 history_days 거래일 뉴스
    hist_news = day.news([symbol], days=history_days).copy()

    # 1) 현재 뉴스 개수
    news_count = len(current_news)

    # 2) 현재 뉴스 평균 tone
    mean_tone = (
        current_news["tone"].mean()
        if len(current_news) > 0
        else np.nan
    )

    # 3) 최근 history_days 거래일의 뉴스 개수
    # 먼저 해당 기간의 실제 거래일 목록을 가져옴
    hist_daily = day.daily(days=history_days).copy()

    trading_dates = (
        pd.to_datetime(hist_daily["date_et"])
        .dt.date
        .drop_duplicates()
        .sort_values()
    )

    if len(hist_news) > 0:
        hist_news["date_et"] = pd.to_datetime(
            hist_news["known_at"]
        ).dt.date

        daily_counts = (
            hist_news
            .groupby("date_et")
            .size()
        )

        # 뉴스가 없는 거래일은 0으로 채우기
        daily_counts = (
            daily_counts
            .reindex(trading_dates, fill_value=0)
        )
    else:
        # 최근 기간 동안 뉴스가 아예 없었던 경우
        daily_counts = pd.Series(
            0,
            index=trading_dates,
            dtype=float
        )

    median_count = daily_counts.median()

    news_count_ratio = (
        news_count / median_count
        if median_count > 0
        else np.nan
    )

    return {
        "news_count": news_count,
        "news_count_ratio": news_count_ratio,
        "mean_tone": mean_tone
    }

In [33]:
get_news_features(day, "AMD")

{'news_count': 56,
 'news_count_ratio': np.float64(1.75),
 'mean_tone': np.float64(1.6427073374045382)}

최근 20거래일 동안 뉴스가 없던 날까지 포함한 평소 수준과 비교했을 때 현재 뉴스량이 약 1.75배라는 뜻

In [36]:
train, test = ds.split("2026-06-01")

print("train length:", len(train))
print("test length :", len(test))
print("total       :", len(train) + len(test))

print("first train date:", train[0].date)
print("first test date :", test[0].date)

train length: 449
test length : 73
total       : 522
first train date: 2024-08-13 00:00:00
first test date : 2026-05-29 00:00:00


In [38]:
news = ds.table("news").copy()

news["symbols"] = news["symbols"].str.split(",")

news_exploded = (
    news
    .explode("symbols")
    .rename(columns={"symbols": "symbol"})
    .copy()
)

news_exploded["symbol"] = news_exploded["symbol"].str.strip()
news_exploded["known_at"] = pd.to_datetime(news_exploded["known_at"])

print(news_exploded.shape)
news_exploded.head()

(4397626, 9)


,symbol,source,url,title,tone,positive,negative,known_at,text
0,MS,nbcnewyork.com,https://www.nbcnewyork.com/news/business/money...,Stock futures are little changed ahead of key ...,0.299401,2.095808,1.796407,2024-08-13 20:00:00,NaN
0,WFC,nbcnewyork.com,https://www.nbcnewyork.com/news/business/money...,Stock futures are little changed ahead of key ...,0.299401,2.095808,1.796407,2024-08-13 20:00:00,NaN
1,AVGO,investorplace.com,https://investorplace.com/market360/2024/08/te...,Tech Earnings Alert! Broadcom Stock May Be the...,2.355072,3.442029,1.086957,2024-08-13 20:00:00,NaN
2,NFLX,cairnspost.com.au,https://www.cairnspost.com.au/lifestyle/when-i...,Where Emily in Paris goes to in Rome in season 4,1.234568,3.254770,2.020202,2024-08-13 20:00:00,NaN
3,GS,fox5ny.com,https://www.fox5ny.com/news/fraud-trial-george...,Fraud trial of George Santos to begin next mon...,-5.429293,1.136364,6.565657,2024-08-13 20:00:00,NaN


수정: normalized news count = 오늘의 뉴스량 % 직전 20개 예측일의 평소 뉴스량 median
왜 수정? 현재까지 포함한 20일 median을 쓰면 오늘 값이 자기 자신을 normalize하는 기준에 들어감 

In [39]:
train, test = ds.split("2026-06-01")

all_days = []

for part in [train, test]:
    for i in range(len(part)):
        d = part[i]

        all_days.append({
            "date_et": pd.to_datetime(d.date),
            "target": pd.to_datetime(d.target),
            "cutoff": pd.to_datetime(d.cutoff)
        })

day_meta = (
    pd.DataFrame(all_days)
    .drop_duplicates("date_et")
    .sort_values("date_et")
    .reset_index(drop=True)
)

print(day_meta.shape)
day_meta.head()

(522, 3)


,date_et,target,cutoff
0,2024-08-13,2024-08-14,2024-08-14 09:30:00
1,2024-08-14,2024-08-15,2024-08-15 09:30:00
2,2024-08-15,2024-08-16,2024-08-16 09:30:00
3,2024-08-16,2024-08-19,2024-08-19 09:30:00
4,2024-08-19,2024-08-20,2024-08-20 09:30:00


In [40]:
rows = []

for _, row in day_meta.iterrows():
    date_et = row["date_et"]
    cutoff = row["cutoff"]

    # 이전 기준일 cutoff 이후 ~ 현재 cutoff까지 뉴스
    prev_cutoff = (
        day_meta.loc[day_meta["date_et"] < date_et, "cutoff"].max()
    )

    if pd.isna(prev_cutoff):
        mask = news_exploded["known_at"] <= cutoff
    else:
        mask = (
            (news_exploded["known_at"] > prev_cutoff)
            & (news_exploded["known_at"] <= cutoff)
        )

    temp = news_exploded.loc[
        mask,
        ["symbol", "tone"]
    ]

    agg = (
        temp.groupby("symbol")
        .agg(
            news_count=("symbol", "size"),
            mean_tone=("tone", "mean")
        )
        .reset_index()
    )

    agg["date_et"] = date_et

    rows.append(agg)

news_features = pd.concat(rows, ignore_index=True)

news_features.head()

,symbol,news_count,mean_tone,date_et
0,AAPL,35,0.540646,2024-08-13
1,ABBV,1,0.873016,2024-08-13
2,ABT,4,1.260202,2024-08-13
3,ADI,3,1.738864,2024-08-13
4,AMAT,4,0.568553,2024-08-13


In [41]:
symbols = sorted(daily["symbol"].unique())

full_index = (
    pd.MultiIndex
    .from_product(
        [day_meta["date_et"], symbols],
        names=["date_et", "symbol"]
    )
    .to_frame(index=False)
)

news_features = full_index.merge(
    news_features,
    on=["date_et", "symbol"],
    how="left"
)

# 뉴스가 없던 날은 count = 0
news_features["news_count"] = (
    news_features["news_count"]
    .fillna(0)
)

print(news_features.shape)

news_features.head(20)

(26100, 4)


,date_et,symbol,news_count,mean_tone
0,2024-08-13,ADI,3.0,1.738864
1,2024-08-13,AMD,12.0,0.651505
2,2024-08-13,AMGN,7.0,-0.417956
3,2024-08-13,AMZN,0.0,NaN
4,2024-08-13,AVGO,21.0,1.314382
5,2024-08-13,AXP,23.0,2.781329
6,2024-08-13,BA,243.0,0.076457
7,2024-08-13,BKNG,11.0,-1.057047
8,2024-08-13,BRK-B,29.0,-0.048572
9,2024-08-13,CB,10.0,2.136392


-> 뉴스가 0개였던 종목/날짜도 채워서 50종목 X 522일 전체 격자 만들기

In [42]:
news_features = (
    news_features
    .sort_values(["symbol", "date_et"])
    .copy()
)

# 오늘을 제외한 직전 20개 예측일의 news_count median
news_features["median_news_count_20d"] = (
    news_features
    .groupby("symbol")["news_count"]
    .transform(
        lambda x: x.shift(1).rolling(
            20,
            min_periods=5
        ).median()
    )
)

# normalized news count
news_features["news_count_ratio"] = (
    news_features["news_count"]
    / news_features["median_news_count_20d"]
)

# 과거 median이 0이면 나눌 수 없으니 NaN 처리
news_features.loc[
    news_features["median_news_count_20d"] == 0,
    "news_count_ratio"
] = np.nan

news_features[
    [
        "date_et",
        "symbol",
        "news_count",
        "median_news_count_20d",
        "news_count_ratio",
        "mean_tone"
    ]
].head(30)

,date_et,symbol,news_count,median_news_count_20d,news_count_ratio,mean_tone
0,2024-08-13,ADI,3.0,NaN,NaN,1.738864
50,2024-08-14,ADI,8.0,NaN,NaN,2.281307
100,2024-08-15,ADI,9.0,NaN,NaN,0.991722
150,2024-08-16,ADI,30.0,NaN,NaN,0.658372
200,2024-08-19,ADI,9.0,NaN,NaN,-0.130245
250,2024-08-20,ADI,42.0,9.0,4.666667,0.300401
300,2024-08-21,ADI,26.0,9.0,2.888889,1.437755
350,2024-08-22,ADI,16.0,9.0,1.777778,1.414980
400,2024-08-23,ADI,15.0,12.5,1.200000,1.176718
450,2024-08-26,ADI,11.0,15.0,0.733333,1.828307


-> 핵심 : shift(1) -> 오늘 뉴스량은 기준 median 계산에서 빼고 오직 과거 데이터만 사용

In [43]:
daily_merge = daily.copy()

daily_merge["date_et"] = pd.to_datetime(daily_merge["date_et"])

news_analysis = daily_merge.merge(
    news_features[
        [
            "date_et",
            "symbol",
            "news_count",
            "news_count_ratio",
            "mean_tone"
        ]
    ],
    on=["date_et", "symbol"],
    how="left"
)

news_analysis[
    [
        "date_et",
        "symbol",
        "next_ret",
        "next_label",
        "news_count",
        "news_count_ratio",
        "mean_tone"
    ]
].head()

,date_et,symbol,next_ret,next_label,news_count,news_count_ratio,mean_tone
0,2023-10-20,ADI,-0.015855,1.0,NaN,NaN,NaN
1,2023-10-23,ADI,0.006469,3.0,NaN,NaN,NaN
2,2023-10-24,ADI,-0.031286,0.0,NaN,NaN,NaN
3,2023-10-25,ADI,0.006384,3.0,NaN,NaN,NaN
4,2023-10-26,ADI,-0.001306,2.0,NaN,NaN,NaN


In [44]:
features = [
    "news_count",
    "news_count_ratio",
    "mean_tone"
]

print("[Spearman correlation with next-day return]")

for col in features:
    temp = news_analysis[[col, "next_ret"]].dropna()

    corr = temp[col].corr(
        temp["next_ret"],
        method="spearman"
    )

    print(f"{col:18s}: {corr:.4f}")

[Spearman correlation with next-day return]
news_count        : -0.0032
news_count_ratio  : 0.0014
mean_tone         : -0.0003


In [45]:
news_analysis["abs_next_ret"] = news_analysis["next_ret"].abs()

print("[Spearman correlation with absolute next-day return]")

for col in features:
    temp = news_analysis[[col, "abs_next_ret"]].dropna()

    corr = temp[col].corr(
        temp["abs_next_ret"],
        method="spearman"
    )

    print(f"{col:18s}: {corr:.4f}")

[Spearman correlation with absolute next-day return]
news_count        : 0.0814
news_count_ratio  : 0.0366
mean_tone         : 0.0106


-> 셋 다 다음 날 상승/하락 방향과는 거의 관계가 없음

In [46]:
def news_quintile_analysis(df, feature):
    temp = df[
        [feature, "next_ret", "next_label"]
    ].dropna().copy()

    temp["q"] = pd.qcut(
        temp[feature],
        5,
        labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
        duplicates="drop"
    )

    result = temp.groupby("q", observed=True).agg(
        count=("next_ret", "size"),
        mean_next_ret=("next_ret", "mean"),
        up_rate=("next_ret", lambda x: (x > 0).mean()),
        big_move_rate=(
            "next_label",
            lambda x: x.isin([0, 4]).mean()
        )
    )

    return result


for feature in [
    "news_count",
    "news_count_ratio",
    "mean_tone"
]:
    print(f"\n===== {feature} =====")
    display(
        news_quintile_analysis(
            news_analysis,
            feature
        )
    )


===== news_count =====


,count,mean_next_ret,up_rate,big_move_rate
q,,,,
Q1,5550,0.000844,0.528108,0.100360
Q2,4949,0.000806,0.522530,0.149121
Q3,5248,0.001240,0.519627,0.161395
Q4,5138,0.001182,0.523161,0.163293
Q5,5215,0.001088,0.518121,0.210163



===== news_count_ratio =====


,count,mean_next_ret,up_rate,big_move_rate
q,,,,
Q1,4539,0.000829,0.516854,0.141221
Q2,4460,0.000824,0.518161,0.169955
Q3,4500,0.000939,0.529111,0.164444
Q4,4505,0.001659,0.528524,0.179800
Q5,4494,0.000974,0.512016,0.185581



===== mean_tone =====


,count,mean_next_ret,up_rate,big_move_rate
q,,,,
Q1,4448,0.000806,0.522032,0.153103
Q2,4448,0.001375,0.525629,0.165243
Q3,4448,0.000664,0.515962,0.183678
Q4,4448,0.001125,0.518210,0.180306
Q5,4448,0.001304,0.519784,0.161421


news_count는 꽤 의미 있음 -> 뉴스가 많이 나온 종목일수록 다음날 급등/급락 가능성 높아지는 경향 (방향보다는 변동 크기 예측용)
news_count ratio :비슷하긴 한데 더 약함
mean_tone -> 우선순위 낮음

In [47]:
earnings = ds.table("earnings").copy()

print("shape:", earnings.shape)
print(earnings.columns.tolist())

earnings.head(10)

shape: (1200, 5)
['symbol', 'known_at', 'eps_estimate', 'eps_reported', 'surprise_pct']


,symbol,known_at,eps_estimate,eps_reported,surprise_pct
0,COST,2020-09-24 16:00:00,2.81,3.13,11.26
1,JNJ,2020-10-13 06:00:00,1.98,2.20,11.03
2,GS,2020-10-14 07:00:00,5.50,9.68,75.85
3,MS,2020-10-15 07:00:00,1.27,1.59,25.42
4,PM,2020-10-20 06:00:00,1.36,1.42,4.45
5,PLD,2020-10-20 08:00:00,0.31,0.40,28.46
6,IBKR,2020-10-20 16:00:00,0.14,0.13,-3.34
7,NFLX,2020-10-20 17:00:00,0.21,0.17,-18.33
8,TMO,2020-10-21 06:00:00,4.30,5.63,30.82
9,VZ,2020-10-21 07:00:00,1.21,1.25,2.90


In [48]:
earnings["known_at"] = pd.to_datetime(earnings["known_at"])

earnings_rows = []

for symbol in symbols:
    # 이 종목의 522개 예측일
    left = (
        day_meta[["date_et", "cutoff"]]
        .copy()
        .sort_values("cutoff")
    )
    left["symbol"] = symbol

    # 이 종목의 실적 발표 기록
    right = (
        earnings[earnings["symbol"] == symbol]
        .copy()
        .sort_values("known_at")
    )

    # cutoff 이전 가장 최근 실적 발표 붙이기
    merged = pd.merge_asof(
        left,
        right,
        left_on="cutoff",
        right_on="known_at",
        direction="backward"
    )

    # symbol 컬럼 정리
    merged["symbol"] = symbol

    earnings_rows.append(merged)

earnings_features = pd.concat(
    earnings_rows,
    ignore_index=True
)

# 실적 발표 후 얼마나 지났는지
earnings_features["days_since_earnings"] = (
    earnings_features["cutoff"]
    - earnings_features["known_at"]
).dt.total_seconds() / (60 * 60 * 24)

earnings_features[
    [
        "date_et",
        "symbol",
        "cutoff",
        "known_at",
        "surprise_pct",
        "days_since_earnings"
    ]
].head(20)

,date_et,symbol,cutoff,known_at,surprise_pct,days_since_earnings
0,2024-08-13,ADI,2024-08-14 09:30:00,2024-05-22 07:00:00,10.34,84.104167
1,2024-08-14,ADI,2024-08-15 09:30:00,2024-05-22 07:00:00,10.34,85.104167
2,2024-08-15,ADI,2024-08-16 09:30:00,2024-05-22 07:00:00,10.34,86.104167
3,2024-08-16,ADI,2024-08-19 09:30:00,2024-05-22 07:00:00,10.34,89.104167
4,2024-08-19,ADI,2024-08-20 09:30:00,2024-05-22 07:00:00,10.34,90.104167
5,2024-08-20,ADI,2024-08-21 09:30:00,2024-08-21 07:00:00,4.83,0.104167
6,2024-08-21,ADI,2024-08-22 09:30:00,2024-08-21 07:00:00,4.83,1.104167
7,2024-08-22,ADI,2024-08-23 09:30:00,2024-08-21 07:00:00,4.83,2.104167
8,2024-08-23,ADI,2024-08-26 09:30:00,2024-08-21 07:00:00,4.83,5.104167
9,2024-08-26,ADI,2024-08-27 09:30:00,2024-08-21 07:00:00,4.83,6.104167


In [49]:
earnings_features["earnings_recent_3d"] = (
    earnings_features["days_since_earnings"] <= 3
).astype(int)

earnings_features[
    [
        "date_et",
        "symbol",
        "surprise_pct",
        "days_since_earnings",
        "earnings_recent_3d"
    ]
].head(20)

,date_et,symbol,surprise_pct,days_since_earnings,earnings_recent_3d
0,2024-08-13,ADI,10.34,84.104167,0
1,2024-08-14,ADI,10.34,85.104167,0
2,2024-08-15,ADI,10.34,86.104167,0
3,2024-08-16,ADI,10.34,89.104167,0
4,2024-08-19,ADI,10.34,90.104167,0
5,2024-08-20,ADI,4.83,0.104167,1
6,2024-08-21,ADI,4.83,1.104167,1
7,2024-08-22,ADI,4.83,2.104167,1
8,2024-08-23,ADI,4.83,5.104167,0
9,2024-08-26,ADI,4.83,6.104167,0


In [50]:
earnings_analysis = daily_merge.merge(
    earnings_features[
        [
            "date_et",
            "symbol",
            "surprise_pct",
            "days_since_earnings",
            "earnings_recent_3d"
        ]
    ],
    on=["date_et", "symbol"],
    how="left"
)

earnings_analysis["abs_surprise_pct"] = (
    earnings_analysis["surprise_pct"].abs()
)

earnings_analysis["abs_next_ret"] = (
    earnings_analysis["next_ret"].abs()
)

In [51]:
features = [
    "surprise_pct",
    "abs_surprise_pct",
    "days_since_earnings"
]

print("[Spearman with next-day return]")

for col in features:
    temp = earnings_analysis[[col, "next_ret"]].dropna()
    corr = temp[col].corr(
        temp["next_ret"],
        method="spearman"
    )
    print(f"{col:20s}: {corr:.4f}")

print("\n[Spearman with absolute next-day return]")

for col in features:
    temp = earnings_analysis[[col, "abs_next_ret"]].dropna()
    corr = temp[col].corr(
        temp["abs_next_ret"],
        method="spearman"
    )
    print(f"{col:20s}: {corr:.4f}")

[Spearman with next-day return]
surprise_pct        : 0.0129
abs_surprise_pct    : 0.0080
days_since_earnings : 0.0165

[Spearman with absolute next-day return]
surprise_pct        : 0.0389
abs_surprise_pct    : 0.0595
days_since_earnings : -0.0417


 earnings쪽도 방향 예측력은 거의 없음
 절댓값은 아주 약하게 신호 있음

earnings는 이벤트형 feature -> earnings_recent_3d = 1인 날과 아닌 날의 big move rate를 직접 비교하는게 더 적절

In [52]:
earnings_group_result = (
    earnings_analysis
    .dropna(subset=["next_label"])
    .groupby("earnings_recent_3d")
    .agg(
        count=("next_label", "size"),
        mean_next_ret=("next_ret", "mean"),
        up_rate=("next_ret", lambda x: (x > 0).mean()),
        big_move_rate=("next_label", lambda x: x.isin([0, 4]).mean())
    )
)

earnings_group_result

,count,mean_next_ret,up_rate,big_move_rate
earnings_recent_3d,,,,
0.0,25185,0.000949,0.523446,0.146754
1.0,915,0.003310,0.492896,0.416393


In [53]:
earnings_analysis["surprise_group"] = pd.cut(
    earnings_analysis["surprise_pct"],
    bins=[-np.inf, -10, 0, 10, np.inf],
    labels=[
        "large_negative",
        "small_negative",
        "small_positive",
        "large_positive"
    ]
)

earnings_surprise_result = (
    earnings_analysis
    .dropna(subset=["surprise_group", "next_label"])
    .groupby("surprise_group", observed=True)
    .agg(
        count=("next_label", "size"),
        mean_next_ret=("next_ret", "mean"),
        up_rate=("next_ret", lambda x: (x > 0).mean()),
        big_move_rate=("next_label", lambda x: x.isin([0, 4]).mean())
    )
)

earnings_surprise_result

,count,mean_next_ret,up_rate,big_move_rate
surprise_group,,,,
large_negative,987,0.000471,0.502533,0.233029
small_negative,1991,0.000562,0.515319,0.147162
small_positive,15483,0.000901,0.522250,0.143189
large_positive,7540,0.001523,0.526658,0.175862


- earnings_recent_3d → 강한 feature 후보
- days_since_earnings → 보조 feature 후보
- abs_surprise_pct → 보조 feature 후보
- surprise_pct 방향값 자체 → 우선순위 낮음
특히 지금까지 본 것 중:
- vol_5d Q1→Q5 big move: 6.2% → 28.4%
- news_count Q1→Q5 big move: 10.0% → 21.0%
- earnings_recent_3d: 14.7% → 41.6%
!!! 결론: earnings_recent_3d가 지금까지 가장 강한 extreme-move signal

In [54]:
analyst = ds.table("analyst").copy()

print("shape:", analyst.shape)
print(analyst.columns.tolist())

analyst.head(10)

shape: (23504, 7)
['symbol', 'known_at', 'firm', 'action', 'target_action', 'target_current', 'target_prior']


,symbol,known_at,firm,action,target_action,target_current,target_prior
0,BA,2012-01-18 18:58:00,Societe Generale,down,NaN,64.0,NaN
1,AMD,2012-02-07 14:24:00,Longbow Research,up,NaN,NaN,NaN
2,MS,2012-02-08 10:29:00,Collins Stewart,down,Raises,23.0,NaN
3,CVS,2012-02-08 12:14:00,Citigroup,main,Raises,51.0,45.0
4,V,2012-02-08 13:05:00,Stifel,main,Raises,125.0,109.0
5,KO,2012-02-08 18:03:00,UBS,main,Raises,73.0,70.0
6,CSCO,2012-02-09 11:40:00,Wunderlich Securities,main,Raises,24.0,22.0
7,V,2012-02-09 11:45:00,Wedbush,main,Raises,90.0,83.0
8,CSCO,2012-02-09 11:57:00,ISI Group,down,Maintains,22.0,NaN
9,V,2012-02-09 12:05:00,Citigroup,main,Raises,115.0,104.0


In [55]:
analyst["known_at"] = pd.to_datetime(analyst["known_at"])

analyst["target_change_pct"] = (
    (analyst["target_current"] - analyst["target_prior"])
    / analyst["target_prior"]
)

analyst[
    [
        "symbol",
        "known_at",
        "action",
        "target_action",
        "target_current",
        "target_prior",
        "target_change_pct"
    ]
].head(20)

,symbol,known_at,action,target_action,target_current,target_prior,target_change_pct
0,BA,2012-01-18 18:58:00,down,NaN,64.0,NaN,NaN
1,AMD,2012-02-07 14:24:00,up,NaN,NaN,NaN,NaN
2,MS,2012-02-08 10:29:00,down,Raises,23.0,NaN,NaN
3,CVS,2012-02-08 12:14:00,main,Raises,51.0,45.0,0.133333
4,V,2012-02-08 13:05:00,main,Raises,125.0,109.0,0.146789
5,KO,2012-02-08 18:03:00,main,Raises,73.0,70.0,0.042857
6,CSCO,2012-02-09 11:40:00,main,Raises,24.0,22.0,0.090909
7,V,2012-02-09 11:45:00,main,Raises,90.0,83.0,0.084337
8,CSCO,2012-02-09 11:57:00,down,Maintains,22.0,NaN,NaN
9,V,2012-02-09 12:05:00,main,Raises,115.0,104.0,0.105769


In [56]:
print("[action]")
print(analyst["action"].value_counts(dropna=False))

print("\n[target_action]")
print(analyst["target_action"].value_counts(dropna=False))

[action]
action
main    17043
up       1783
init     1608
down     1554
reit     1516
Name: count, dtype: int64

[target_action]
target_action
Raises       12297
Lowers        5530
Announces     2134
Maintains     1885
NaN           1600
Removes         31
Adjusts         26
raises           1
Name: count, dtype: int64


In [57]:
analyst["action"] = analyst["action"].str.lower()
analyst["target_action"] = analyst["target_action"].str.lower()

analyst["action_up"] = (analyst["action"] == "up").astype(int)
analyst["action_down"] = (analyst["action"] == "down").astype(int)

analyst["target_raise"] = (
    analyst["target_action"] == "raises"
).astype(int)

analyst["target_lower"] = (
    analyst["target_action"] == "lowers"
).astype(int)

analyst[
    [
        "symbol",
        "known_at",
        "action",
        "target_action",
        "target_change_pct",
        "action_up",
        "action_down",
        "target_raise",
        "target_lower"
    ]
].head(20)

,symbol,known_at,action,target_action,target_change_pct,action_up,action_down,target_raise,target_lower
0,BA,2012-01-18 18:58:00,down,NaN,NaN,0,1,0,0
1,AMD,2012-02-07 14:24:00,up,NaN,NaN,1,0,0,0
2,MS,2012-02-08 10:29:00,down,raises,NaN,0,1,1,0
3,CVS,2012-02-08 12:14:00,main,raises,0.133333,0,0,1,0
4,V,2012-02-08 13:05:00,main,raises,0.146789,0,0,1,0
5,KO,2012-02-08 18:03:00,main,raises,0.042857,0,0,1,0
6,CSCO,2012-02-09 11:40:00,main,raises,0.090909,0,0,1,0
7,V,2012-02-09 11:45:00,main,raises,0.084337,0,0,1,0
8,CSCO,2012-02-09 11:57:00,down,maintains,NaN,0,1,0,0
9,V,2012-02-09 12:05:00,main,raises,0.105769,0,0,1,0


In [58]:
analyst_rows = []

for symbol in symbols:
    left = (
        day_meta[["date_et", "cutoff"]]
        .copy()
        .sort_values("cutoff")
    )
    left["symbol"] = symbol

    right = (
        analyst[analyst["symbol"] == symbol]
        .copy()
        .sort_values("known_at")
    )

    merged = pd.merge_asof(
        left,
        right,
        left_on="cutoff",
        right_on="known_at",
        direction="backward"
    )

    merged["symbol"] = symbol

    analyst_rows.append(merged)

analyst_features = pd.concat(
    analyst_rows,
    ignore_index=True
)

analyst_features["days_since_analyst"] = (
    analyst_features["cutoff"]
    - analyst_features["known_at"]
).dt.total_seconds() / (60 * 60 * 24)

analyst_features["analyst_recent_3d"] = (
    analyst_features["days_since_analyst"] <= 3
).astype(int)

analyst_features[
    [
        "date_et",
        "symbol",
        "days_since_analyst",
        "analyst_recent_3d",
        "action_up",
        "action_down",
        "target_raise",
        "target_lower",
        "target_change_pct"
    ]
].head(20)

,date_et,symbol,days_since_analyst,analyst_recent_3d,action_up,action_down,target_raise,target_lower,target_change_pct
0,2024-08-13,ADI,26.764514,0,0.0,0.0,1.0,0.0,0.06383
1,2024-08-14,ADI,27.764514,0,0.0,0.0,1.0,0.0,0.06383
2,2024-08-15,ADI,28.764514,0,0.0,0.0,1.0,0.0,0.06383
3,2024-08-16,ADI,31.764514,0,0.0,0.0,1.0,0.0,0.06383
4,2024-08-19,ADI,32.764514,0,0.0,0.0,1.0,0.0,0.06383
5,2024-08-20,ADI,33.764514,0,0.0,0.0,1.0,0.0,0.06383
6,2024-08-21,ADI,34.764514,0,0.0,0.0,1.0,0.0,0.06383
7,2024-08-22,ADI,0.647905,1,0.0,0.0,1.0,0.0,0.02800
8,2024-08-23,ADI,3.647905,0,0.0,0.0,1.0,0.0,0.02800
9,2024-08-26,ADI,4.647905,0,0.0,0.0,1.0,0.0,0.02800


In [59]:
analyst_analysis = daily_merge.merge(
    analyst_features[
        [
            "date_et",
            "symbol",
            "days_since_analyst",
            "analyst_recent_3d",
            "action_up",
            "action_down",
            "target_raise",
            "target_lower",
            "target_change_pct"
        ]
    ],
    on=["date_et", "symbol"],
    how="left"
)

analyst_analysis["abs_target_change_pct"] = (
    analyst_analysis["target_change_pct"].abs()
)

analyst_analysis["abs_next_ret"] = (
    analyst_analysis["next_ret"].abs()
)

In [60]:
features = [
    "target_change_pct",
    "abs_target_change_pct",
    "days_since_analyst"
]

print("[Spearman with next-day return]")

for col in features:
    temp = analyst_analysis[[col, "next_ret"]].dropna()
    corr = temp[col].corr(
        temp["next_ret"],
        method="spearman"
    )
    print(f"{col:24s}: {corr:.4f}")

print("\n[Spearman with absolute next-day return]")

for col in features:
    temp = analyst_analysis[[col, "abs_next_ret"]].dropna()
    corr = temp[col].corr(
        temp["abs_next_ret"],
        method="spearman"
    )
    print(f"{col:24s}: {corr:.4f}")

[Spearman with next-day return]
target_change_pct       : -0.0025
abs_target_change_pct   : 0.0125
days_since_analyst      : -0.0061

[Spearman with absolute next-day return]
target_change_pct       : -0.0078
abs_target_change_pct   : 0.0654
days_since_analyst      : -0.0512


In [61]:
analyst_recent_result = (
    analyst_analysis
    .dropna(subset=["next_label"])
    .groupby("analyst_recent_3d")
    .agg(
        count=("next_label", "size"),
        mean_next_ret=("next_ret", "mean"),
        up_rate=("next_ret", lambda x: (x > 0).mean()),
        big_move_rate=("next_label", lambda x: x.isin([0, 4]).mean())
    )
)

analyst_recent_result

,count,mean_next_ret,up_rate,big_move_rate
analyst_recent_3d,,,,
0.0,20140,0.00081,0.521351,0.147319
1.0,5960,0.00178,0.525839,0.186242


약한 signal은 있지만 earnings보다 훨씬 약함
-> analyst event 직후에는 평소보다 급등락 가능성이 조금 높아짐

현재까지 우선순위는 대충:
earnings_recent_3d > vol_5d > news_count > analyst_recent_3d

In [62]:
group_features = [
    "action_up",
    "action_down",
    "target_raise",
    "target_lower"
]

for col in group_features:
    print(f"\n===== {col} =====")
    
    result = (
        analyst_analysis
        .dropna(subset=["next_label", col])
        .groupby(col)
        .agg(
            count=("next_label", "size"),
            mean_next_ret=("next_ret", "mean"),
            up_rate=("next_ret", lambda x: (x > 0).mean()),
            big_move_rate=("next_label", lambda x: x.isin([0, 4]).mean())
        )
    )

    display(result)


===== action_up =====


,count,mean_next_ret,up_rate,big_move_rate
action_up,,,,
0.0,24267,0.001015,0.521902,0.157251
1.0,1776,0.001285,0.527027,0.146396



===== action_down =====


,count,mean_next_ret,up_rate,big_move_rate
action_down,,,,
0.0,24362,0.001021,0.520893,0.158731
1.0,1681,0.001209,0.541939,0.124331



===== target_raise =====


,count,mean_next_ret,up_rate,big_move_rate
target_raise,,,,
0.0,11099,0.001126,0.517074,0.172358
1.0,14944,0.000964,0.526097,0.144740



===== target_lower =====


,count,mean_next_ret,up_rate,big_move_rate
target_lower,,,,
0.0,20038,0.000900,0.522657,0.150664
1.0,6005,0.001479,0.520899,0.176020


-> 현재 오래된 raise까지 포함하고 있음 -> analyst_recent_3d == 1 : 즉 3일 안에 analyst event가 있었던 경우만 따로 뽑아서 비교해야함

In [63]:
recent_analyst = analyst_analysis[
    analyst_analysis["analyst_recent_3d"] == 1
].copy()

group_features = [
    "action_up",
    "action_down",
    "target_raise",
    "target_lower"
]

for col in group_features:
    print(f"\n===== {col} (recent 3d only) =====")

    result = (
        recent_analyst
        .dropna(subset=["next_label", col])
        .groupby(col)
        .agg(
            count=("next_label", "size"),
            mean_next_ret=("next_ret", "mean"),
            up_rate=("next_ret", lambda x: (x > 0).mean()),
            big_move_rate=("next_label", lambda x: x.isin([0, 4]).mean())
        )
    )

    display(result)


===== action_up (recent 3d only) =====


,count,mean_next_ret,up_rate,big_move_rate
action_up,,,,
0.0,5609,0.001598,0.523979,0.188269
1.0,351,0.004686,0.555556,0.153846



===== action_down (recent 3d only) =====


,count,mean_next_ret,up_rate,big_move_rate
action_down,,,,
0.0,5692,0.001748,0.524772,0.186578
1.0,268,0.002447,0.548507,0.179104



===== target_raise (recent 3d only) =====


,count,mean_next_ret,up_rate,big_move_rate
target_raise,,,,
0.0,2691,0.002114,0.526198,0.209959
1.0,3269,0.001505,0.525543,0.166718



===== target_lower (recent 3d only) =====


,count,mean_next_ret,up_rate,big_move_rate
target_lower,,,,
0.0,4506,0.001664,0.524634,0.171549
1.0,1454,0.002139,0.529574,0.231774


analyst_recent_3d → 유지 후보, big move 14.7% → 18.6%
abs_target_change_pct → 유지 후보, magnitude와 약한 양의 관계
target_lower → 보조 후보, 최근 3일 내 lower면 big move 23.2%
action_up, action_down, target_raise → 방향 예측력 거의 없음

모델링을 어떻게 헤야하는가?
1단계: 방향 예측
down / neutral / up

2단계: 움직임 크기 예측
normal / extreme

→ 합쳐서 0~4 label 결정

In [64]:
price = ds.table("price").copy()

print("shape:", price.shape)
print(price.columns.tolist())

print("\n[session]")
print(price["session"].value_counts(dropna=False))

price.head(15)

shape: (590024, 9)
['symbol', 'datetime', 'open', 'high', 'low', 'close', 'volume', 'session', 'known_at']

[session]
session
regular    253208
pre        207133
post       129683
Name: count, dtype: int64


,symbol,datetime,open,high,low,close,volume,session,known_at
0,CSCO,2023-10-20 04:00:00,53.320,53.320000,53.320000,53.320,0,pre,2023-10-20 05:00:00
1,NFLX,2023-10-20 04:00:00,40.027,40.184000,39.773003,40.150,0,pre,2023-10-20 05:00:00
2,NOW,2023-10-20 04:00:00,111.338,112.076004,111.228004,111.954,0,pre,2023-10-20 05:00:00
3,ORCL,2023-10-20 04:00:00,108.180,108.180000,107.830000,107.830,0,pre,2023-10-20 05:00:00
4,PFE,2023-10-20 04:00:00,31.130,31.200000,31.120000,31.190,0,pre,2023-10-20 05:00:00
5,UBER,2023-10-20 04:00:00,42.550,42.570000,42.070000,42.570,0,pre,2023-10-20 05:00:00
6,PM,2023-10-20 04:00:00,90.720,90.720000,90.720000,90.720,0,pre,2023-10-20 05:00:00
7,RTX,2023-10-20 04:00:00,74.370,74.460000,74.250000,74.330,0,pre,2023-10-20 05:00:00
8,NEM,2023-10-20 04:00:00,39.500,39.570000,39.440000,39.570,0,pre,2023-10-20 05:00:00
9,UNP,2023-10-20 04:00:00,210.330,212.390000,210.330000,212.390,0,pre,2023-10-20 05:00:00


In [65]:
price["datetime"] = pd.to_datetime(price["datetime"])
price["known_at"] = pd.to_datetime(price["known_at"])

print("datetime min/max:")
print(price["datetime"].min(), "->", price["datetime"].max())

print("\nknown_at min/max:")
print(price["known_at"].min(), "->", price["known_at"].max())

datetime min/max:
2023-10-20 04:00:00 -> 2026-09-14 19:00:00

known_at min/max:
2023-10-20 05:00:00 -> 2026-09-14 20:00:00


session이 pre / regular / post로 구분돼 있고 known_at도 있어서 cutoff 이전 데이터만 안전하게 쓸 수 있음
pre/post의 volume=0은 이 데이터 특성이라 volume은 크게 신뢰하지 말고 가격 움직임 위주로 봐야함

!!! premarket_move
= target-day 09:30 직전까지 알 수 있는 마지막 pre-market 가격 / 이전 regular session의 마지막 종가- 1

In [66]:
sample_day = ds.day("2026-02-20")

sample_price = sample_day.price(days=1).copy()

print("base date :", sample_day.date)
print("target    :", sample_day.target)
print("cutoff    :", sample_day.cutoff)

sample_price[
    ["symbol", "datetime", "close", "session", "known_at"]
].head(30)

base date : 2026-02-20 00:00:00
target    : 2026-02-23 00:00:00
cutoff    : 2026-02-23 09:30:00


,symbol,datetime,close,session,known_at
0,NEM,2026-02-20 04:00:00,121.05,pre,2026-02-20 05:00:00
1,MCD,2026-02-20 04:00:00,327.50,pre,2026-02-20 05:00:00
2,UBER,2026-02-20 04:00:00,73.28,pre,2026-02-20 05:00:00
3,VZ,2026-02-20 04:00:00,48.61,pre,2026-02-20 05:00:00
4,AMD,2026-02-20 04:00:00,204.48,pre,2026-02-20 05:00:00
5,PM,2026-02-20 04:00:00,183.60,pre,2026-02-20 05:00:00
6,BRK-B,2026-02-20 04:00:00,497.52,pre,2026-02-20 05:00:00
7,QCOM,2026-02-20 04:00:00,141.76,pre,2026-02-20 05:00:00
8,WMT,2026-02-20 04:00:00,125.04,pre,2026-02-20 05:00:00
9,COP,2026-02-20 04:00:00,112.00,pre,2026-02-20 05:00:00


In [67]:
symbol = "AMD"

amd_price = sample_price[
    sample_price["symbol"] == symbol
].copy()

display(
    amd_price[
        ["datetime", "close", "session", "known_at"]
    ]
)

,datetime,close,session,known_at
4,2026-02-20 04:00:00,204.480000,pre,2026-02-20 05:00:00
77,2026-02-20 05:00:00,204.000000,pre,2026-02-20 06:00:00
107,2026-02-20 06:00:00,203.310000,pre,2026-02-20 07:00:00
183,2026-02-20 07:00:00,202.100000,pre,2026-02-20 08:00:00
239,2026-02-20 08:00:00,202.580000,pre,2026-02-20 09:00:00
263,2026-02-20 09:00:00,200.020000,pre,2026-02-20 09:30:00
324,2026-02-20 09:30:00,202.319901,regular,2026-02-20 10:30:00
390,2026-02-20 10:30:00,201.917007,regular,2026-02-20 11:30:00
400,2026-02-20 11:30:00,199.259995,regular,2026-02-20 12:30:00
458,2026-02-20 12:30:00,199.177994,regular,2026-02-20 13:30:00


In [70]:
symbol = "AMD"

amd_price = sample_price[
    sample_price["symbol"] == symbol
].copy()

target_date = pd.to_datetime(sample_day.target).date()
base_date = pd.to_datetime(sample_day.date).date()

# 직전 거래일 regular 마지막 가격
prev_regular = amd_price[
    (amd_price["session"] == "regular") &
    (amd_price["datetime"].dt.date == base_date)
].sort_values("known_at")

prev_close = prev_regular.iloc[-1]["close"]

# 대상일 cutoff 전에 사용 가능한 pre-market 중 가장 최근 가격
target_pre = amd_price[
    (amd_price["session"] == "pre") &
    (amd_price["datetime"].dt.date == target_date) &
    (amd_price["known_at"] <= sample_day.cutoff)
].sort_values("known_at")

pre_last = target_pre.iloc[-1]["close"]

premarket_move = pre_last / prev_close - 1

print("previous regular close :", prev_close)
print("latest pre-market price:", pre_last)
print("premarket move         :", premarket_move)
print("premarket move (%)     :", premarket_move * 100)

previous regular close : 200.1199951171875
latest pre-market price: 197.71
premarket move         : -0.01204275022981205
premarket move (%)     : -1.204275022981205


In [71]:
price = price.copy()

price["datetime"] = pd.to_datetime(price["datetime"])
price["known_at"] = pd.to_datetime(price["known_at"])

premarket_rows = []

for _, row in day_meta.iterrows():
    base_date = row["date_et"].date()
    target_date = row["target"].date()
    cutoff = row["cutoff"]

    # 전 거래일 regular 마지막 가격
    prev_regular = (
        price[
            (price["session"] == "regular") &
            (price["datetime"].dt.date == base_date)
        ]
        .sort_values(["symbol", "known_at"])
        .groupby("symbol")
        .tail(1)
        [["symbol", "close"]]
        .rename(columns={"close": "prev_regular_close"})
    )

    # 대상일 cutoff 전에 알 수 있는 마지막 pre-market 가격
    target_pre = (
        price[
            (price["session"] == "pre") &
            (price["datetime"].dt.date == target_date) &
            (price["known_at"] <= cutoff)
        ]
        .sort_values(["symbol", "known_at"])
        .groupby("symbol")
        .tail(1)
        [["symbol", "close"]]
        .rename(columns={"close": "pre_last"})
    )

    temp = prev_regular.merge(
        target_pre,
        on="symbol",
        how="inner"
    )

    temp["date_et"] = row["date_et"]

    temp["premarket_move"] = (
        temp["pre_last"]
        / temp["prev_regular_close"]
        - 1
    )

    premarket_rows.append(temp)

premarket_features = pd.concat(
    premarket_rows,
    ignore_index=True
)

print(premarket_features.shape)

premarket_features[
    [
        "date_et",
        "symbol",
        "prev_regular_close",
        "pre_last",
        "premarket_move"
    ]
].head(20)

(26027, 5)


,date_et,symbol,prev_regular_close,pre_last,premarket_move
0,2024-08-13,ADI,218.059998,216.3900,-0.007658
1,2024-08-13,AMD,141.179993,143.1950,0.014273
2,2024-08-13,AMGN,323.500000,320.2800,-0.009954
3,2024-08-13,AMZN,170.229996,172.1150,0.011073
4,2024-08-13,AVGO,156.320007,158.0650,0.011163
5,2024-08-13,AXP,238.990005,239.0000,0.000042
6,2024-08-13,BA,168.740005,168.2900,-0.002667
7,2024-08-13,BKNG,142.808792,142.7104,-0.000689
8,2024-08-13,BRK-B,432.489990,433.9295,0.003328
9,2024-08-13,CB,267.859985,268.4700,0.002277


In [72]:
premarket_analysis = daily_merge.merge(
    premarket_features[
        [
            "date_et",
            "symbol",
            "premarket_move"
        ]
    ],
    on=["date_et", "symbol"],
    how="left"
)

In [73]:
temp = premarket_analysis[
    ["premarket_move", "next_ret"]
].dropna()

corr = temp["premarket_move"].corr(
    temp["next_ret"],
    method="spearman"
)

print(
    "Spearman premarket_move vs next_ret:",
    round(corr, 4)
)

Spearman premarket_move vs next_ret: 0.4214


In [74]:
temp = premarket_analysis[
    ["premarket_move", "next_ret", "next_label"]
].dropna().copy()

temp["q"] = pd.qcut(
    temp["premarket_move"],
    5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
)

result = (
    temp.groupby("q", observed=True)
    .agg(
        count=("next_ret", "size"),
        mean_next_ret=("next_ret", "mean"),
        up_rate=("next_ret", lambda x: (x > 0).mean()),
        big_move_rate=("next_label", lambda x: x.isin([0, 4]).mean())
    )
)

result

,count,mean_next_ret,up_rate,big_move_rate
q,,,,
Q1,5206,-0.012328,0.279293,0.257011
Q2,5205,-0.002130,0.428242,0.079731
Q3,5205,0.000666,0.532181,0.065514
Q4,5205,0.003486,0.614025,0.091066
Q5,5206,0.015373,0.755474,0.287553


가장강한 signal !!! -> pre-market이 핵심 feature

기존 premarket_mpove = 전날 regular close -> 오늘 pre-market 마지막 가격
-> post + overnight가 합쳐진 효과

In [75]:
price = price.copy()

price["datetime"] = pd.to_datetime(price["datetime"])
price["known_at"] = pd.to_datetime(price["known_at"])

extended_rows = []

for _, row in day_meta.iterrows():
    base_date = row["date_et"].date()
    target_date = row["target"].date()
    cutoff = row["cutoff"]

    # 1) 전날 regular 마지막 가격
    prev_regular = (
        price[
            (price["session"] == "regular") &
            (price["datetime"].dt.date == base_date)
        ]
        .sort_values(["symbol", "known_at"])
        .groupby("symbol")
        .tail(1)
        [["symbol", "close"]]
        .rename(columns={"close": "prev_regular_close"})
    )

    # 2) 전날 post-market 마지막 가격
    prev_post = (
        price[
            (price["session"] == "post") &
            (price["datetime"].dt.date == base_date)
        ]
        .sort_values(["symbol", "known_at"])
        .groupby("symbol")
        .tail(1)
        [["symbol", "close"]]
        .rename(columns={"close": "prev_post_last"})
    )

    # 3) 대상일 pre-market
    target_pre_all = price[
        (price["session"] == "pre") &
        (price["datetime"].dt.date == target_date) &
        (price["known_at"] <= cutoff)
    ].sort_values(["symbol", "known_at"])

    target_pre_first = (
        target_pre_all
        .groupby("symbol")
        .head(1)
        [["symbol", "close"]]
        .rename(columns={"close": "pre_first"})
    )

    target_pre_last = (
        target_pre_all
        .groupby("symbol")
        .tail(1)
        [["symbol", "close"]]
        .rename(columns={"close": "pre_last"})
    )

    temp = (
        prev_regular
        .merge(prev_post, on="symbol", how="outer")
        .merge(target_pre_first, on="symbol", how="outer")
        .merge(target_pre_last, on="symbol", how="outer")
    )

    temp["date_et"] = row["date_et"]

    # 전날 장 마감 후 움직임
    temp["post_market_move"] = (
        temp["prev_post_last"] / temp["prev_regular_close"] - 1
    )

    # 전날 post 마지막 → 오늘 pre 마지막
    temp["overnight_move"] = (
        temp["pre_last"] / temp["prev_post_last"] - 1
    )

    # 오늘 pre-market 내부 움직임
    temp["premarket_internal_move"] = (
        temp["pre_last"] / temp["pre_first"] - 1
    )

    # 기존 feature
    temp["premarket_move"] = (
        temp["pre_last"] / temp["prev_regular_close"] - 1
    )

    extended_rows.append(temp)

extended_features = pd.concat(
    extended_rows,
    ignore_index=True
)

extended_features.head()

,symbol,prev_regular_close,prev_post_last,pre_first,pre_last,date_et,post_market_move,overnight_move,premarket_internal_move,premarket_move
0,ADI,218.059998,219.95,217.40,216.390,2024-08-13,0.008667,-0.016185,-0.004646,-0.007658
1,AMD,141.179993,141.01,140.90,143.195,2024-08-13,-0.001204,0.015495,0.016288,0.014273
2,AMGN,323.500000,322.00,324.19,320.280,2024-08-13,-0.004637,-0.005342,-0.012061,-0.009954
3,AMZN,170.229996,170.40,170.34,172.115,2024-08-13,0.000999,0.010065,0.010420,0.011073
4,AVGO,156.320007,156.63,155.87,158.065,2024-08-13,0.001983,0.009162,0.014082,0.011163


In [76]:
extended_analysis = daily_merge.merge(
    extended_features[
        [
            "date_et",
            "symbol",
            "post_market_move",
            "overnight_move",
            "premarket_internal_move",
            "premarket_move"
        ]
    ],
    on=["date_et", "symbol"],
    how="left"
)

In [77]:
features = [
    "post_market_move",
    "overnight_move",
    "premarket_internal_move",
    "premarket_move"
]

for col in features:
    temp = extended_analysis[
        [col, "next_ret"]
    ].dropna()

    corr = temp[col].corr(
        temp["next_ret"],
        method="spearman"
    )

    print(f"{col:25s}: {corr:.4f}")

post_market_move         : 0.1127
overnight_move           : 0.3650
premarket_internal_move  : 0.2608
premarket_move           : 0.4214


방향성 정보는 전날 post-market 보다 overnight 구간 + 당일 pre-market 내부 움직임에서 더 강하게 형성됨

In [78]:
features = [
    "post_market_move",
    "overnight_move",
    "premarket_internal_move",
    "premarket_move"
]

for col in features:
    print(f"\n===== {col} =====")

    temp = extended_analysis[
        [col, "next_ret", "next_label"]
    ].dropna().copy()

    temp["q"] = pd.qcut(
        temp[col],
        5,
        labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
    )

    result = (
        temp.groupby("q", observed=True)
        .agg(
            count=("next_ret", "size"),
            mean_next_ret=("next_ret", "mean"),
            up_rate=("next_ret", lambda x: (x > 0).mean()),
            big_move_rate=("next_label", lambda x: x.isin([0, 4]).mean())
        )
    )

    display(result)


===== post_market_move =====


,count,mean_next_ret,up_rate,big_move_rate
q,,,,
Q1,5210,-0.002381,0.467179,0.208829
Q2,5209,-0.000022,0.502592,0.128816
Q3,5209,0.000651,0.522365,0.104819
Q4,5209,0.001382,0.531388,0.121904
Q5,5210,0.005445,0.586180,0.216123



===== overnight_move =====


,count,mean_next_ret,up_rate,big_move_rate
q,,,,
Q1,5205,-0.010531,0.319116,0.238040
Q2,5205,-0.001874,0.444380,0.091643
Q3,5204,0.000774,0.526710,0.070907
Q4,5205,0.002888,0.587704,0.109318
Q5,5205,0.013818,0.731604,0.271085



===== premarket_internal_move =====


,count,mean_next_ret,up_rate,big_move_rate
q,,,,
Q1,5206,-0.007567,0.369766,0.229351
Q2,5205,-0.000865,0.469741,0.108165
Q3,5205,0.001251,0.536792,0.090298
Q4,5205,0.002310,0.565034,0.114890
Q5,5206,0.009938,0.667883,0.238187



===== premarket_move =====


,count,mean_next_ret,up_rate,big_move_rate
q,,,,
Q1,5206,-0.012328,0.279293,0.257011
Q2,5205,-0.002130,0.428242,0.079731
Q3,5205,0.000666,0.532181,0.065514
Q4,5205,0.003486,0.614025,0.091066
Q5,5206,0.015373,0.755474,0.287553


결론: 전날 장 마감 직후의 post-market 움직임 자체보다는, 밤사이 + 당일 pre-market에서 형성된 가격 움직임이 실제 당일 종가 방향을 훨씬 잘 설명한다.

post_market_move
→ 약한 방향 signal

overnight_move
→ 강한 방향 + magnitude signal

premarket_internal_move
→ 중간~강한 방향 signal

premarket_move
→ 가장 강한 종합 signal

In [79]:
price_features = [
    "post_market_move",
    "overnight_move",
    "premarket_internal_move",
    "premarket_move"
]

corr_matrix = (
    extended_analysis[price_features]
    .corr(method="spearman")
)

corr_matrix

,post_market_move,overnight_move,premarket_internal_move,premarket_move
post_market_move,1.000000,-0.162077,0.015861,0.266954
overnight_move,-0.162077,1.000000,0.642729,0.842460
premarket_internal_move,0.015861,0.642729,1.000000,0.629678
premarket_move,0.266954,0.842460,0.629678,1.000000


premarket_move와 overnight_move는 강하게 겹침
premarket_internal_move는 어느 정도 별도 정보를 가짐
post_market_move는 가장 독립적이지만, target 방향과의 관계 자체가 약함

premarket_move → 반드시 유지
premarket_internal_move → 추가 정보 가능성 있어서 유지 후보
overnight_move → 강하지만 premarket_move와 중복 큼
post_market_move → 독립적이지만 약해서 우선순위 낮음

In [80]:
direction_rows = []

for _, row in day_meta.iterrows():
    base_date = row["date_et"].date()
    target_date = row["target"].date()
    cutoff = row["cutoff"]

    target_pre = price[
        (price["session"] == "pre") &
        (price["datetime"].dt.date == target_date) &
        (price["known_at"] <= cutoff)
    ].copy()

    if target_pre.empty:
        continue

    target_pre = target_pre.sort_values(["symbol", "known_at"])

    # 종목별 첫/마지막 pre-market bar
    first_pre = (
        target_pre.groupby("symbol").head(1)
        [["symbol", "close"]]
        .rename(columns={"close": "pre_first"})
    )

    last_pre = (
        target_pre.groupby("symbol").tail(1)
        [["symbol", "close"]]
        .rename(columns={"close": "pre_last"})
    )

    # pre-market 고가/저가
    pre_range = (
        target_pre.groupby("symbol")
        .agg(
            pre_high=("high", "max"),
            pre_low=("low", "min")
        )
        .reset_index()
    )

    # 마지막 2개 사용 가능한 pre-market bar
    last2 = (
        target_pre.groupby("symbol")
        .tail(2)
        .groupby("symbol")
        .agg(
            last2_first=("close", "first"),
            last2_last=("close", "last")
        )
        .reset_index()
    )

    # 전날 regular close
    prev_regular = (
        price[
            (price["session"] == "regular") &
            (price["datetime"].dt.date == base_date)
        ]
        .sort_values(["symbol", "known_at"])
        .groupby("symbol")
        .tail(1)
        [["symbol", "close"]]
        .rename(columns={"close": "prev_regular_close"})
    )

    temp = (
        prev_regular
        .merge(first_pre, on="symbol", how="inner")
        .merge(last_pre, on="symbol", how="inner")
        .merge(pre_range, on="symbol", how="inner")
        .merge(last2, on="symbol", how="inner")
    )

    temp["date_et"] = row["date_et"]

    # 1) 개장 직전 2개 pre bar의 수익률
    temp["last_2h_premarket_return"] = (
        temp["last2_last"] / temp["last2_first"] - 1
    )

    # 2) pre-market range 내 마지막 가격 위치
    temp["premarket_range_position"] = (
        (temp["pre_last"] - temp["pre_low"]) /
        (temp["pre_high"] - temp["pre_low"])
    )

    # 3) 기존 premarket_move
    temp["premarket_move"] = (
        temp["pre_last"] / temp["prev_regular_close"] - 1
    )

    direction_rows.append(temp)

direction_features = pd.concat(direction_rows, ignore_index=True)

# 같은 날짜 전체 종목 median 대비 상대 움직임
direction_features["market_relative_premarket"] = (
    direction_features["premarket_move"]
    - direction_features.groupby("date_et")["premarket_move"].transform("median")
)

direction_features[
    [
        "date_et",
        "symbol",
        "last_2h_premarket_return",
        "premarket_range_position",
        "market_relative_premarket"
    ]
].head()

,date_et,symbol,last_2h_premarket_return,premarket_range_position,market_relative_premarket
0,2024-08-13,ADI,-0.006656,0.000000,-0.008985
1,2024-08-13,AMD,0.006997,0.787324,0.012946
2,2024-08-13,AMGN,-0.006853,0.000000,-0.011280
3,2024-08-13,AMZN,0.008230,0.296209,0.009747
4,2024-08-13,AVGO,0.010161,0.878223,0.009837


In [81]:
direction_analysis = daily_merge.merge(
    direction_features[
        [
            "date_et",
            "symbol",
            "last_2h_premarket_return",
            "premarket_range_position",
            "market_relative_premarket"
        ]
    ],
    on=["date_et", "symbol"],
    how="left"
)

new_direction_features = [
    "last_2h_premarket_return",
    "premarket_range_position",
    "market_relative_premarket"
]

for col in new_direction_features:
    temp = direction_analysis[[col, "next_ret"]].dropna()

    corr = temp[col].corr(
        temp["next_ret"],
        method="spearman"
    )

    print(f"{col:30s}: {corr:.4f}")

last_2h_premarket_return      : 0.1506
premarket_range_position      : 0.2423
market_relative_premarket     : 0.3509


In [82]:
check_features = [
    "premarket_move",
    "last_2h_premarket_return",
    "premarket_range_position",
    "market_relative_premarket"
]

tmp = extended_analysis.merge(
    direction_features[
        [
            "date_et",
            "symbol",
            "last_2h_premarket_return",
            "premarket_range_position",
            "market_relative_premarket"
        ]
    ],
    on=["date_et", "symbol"],
    how="left"
)

tmp[check_features].corr(method="spearman")

,premarket_move,last_2h_premarket_return,premarket_range_position,market_relative_premarket
premarket_move,1.000000,0.434775,0.578407,0.823141
last_2h_premarket_return,0.434775,1.000000,0.546999,0.418312
premarket_range_position,0.578407,0.546999,1.000000,0.540936
market_relative_premarket,0.823141,0.418312,0.540936,1.000000


premarket_move → 필수
premarket_range_position → 유지
market_relative_premarket → 유지 후보. 중복은 높지만 “시장 대비 상대 강도”라는 의미가 있어서 일단 넣어보고 나중에 ablation

In [83]:
final_features = daily_merge[
    [
        "date_et",
        "symbol",
        "ret",
        "ret_5d",
        "vol_5d",
        "next_ret",
        "next_label"
    ]
].copy()

In [84]:
final_features = final_features.merge(
    news_features[
        [
            "date_et",
            "symbol",
            "news_count",
            "news_count_ratio"
        ]
    ],
    on=["date_et", "symbol"],
    how="left"
)

In [85]:
earnings_tmp = earnings_features[
    [
        "date_et",
        "symbol",
        "earnings_recent_3d",
        "surprise_pct"
    ]
].copy()

earnings_tmp["abs_surprise_pct"] = (
    earnings_tmp["surprise_pct"].abs()
)

final_features = final_features.merge(
    earnings_tmp[
        [
            "date_et",
            "symbol",
            "earnings_recent_3d",
            "abs_surprise_pct"
        ]
    ],
    on=["date_et", "symbol"],
    how="left"
)

In [86]:
earnings_tmp = earnings_features[
    [
        "date_et",
        "symbol",
        "earnings_recent_3d",
        "surprise_pct"
    ]
].copy()

earnings_tmp["abs_surprise_pct"] = (
    earnings_tmp["surprise_pct"].abs()
)

final_features = final_features.merge(
    earnings_tmp[
        [
            "date_et",
            "symbol",
            "earnings_recent_3d",
            "abs_surprise_pct"
        ]
    ],
    on=["date_et", "symbol"],
    how="left"
)

In [87]:
analyst_tmp = analyst_features[
    [
        "date_et",
        "symbol",
        "analyst_recent_3d",
        "target_lower",
        "target_change_pct"
    ]
].copy()

analyst_tmp["abs_target_change_pct"] = (
    analyst_tmp["target_change_pct"].abs()
)

final_features = final_features.merge(
    analyst_tmp[
        [
            "date_et",
            "symbol",
            "analyst_recent_3d",
            "target_lower",
            "abs_target_change_pct"
        ]
    ],
    on=["date_et", "symbol"],
    how="left"
)

In [90]:
vars_to_check = [
    "daily_merge",
    "news_features",
    "earnings_features",
    "analyst_features",
    "extended_features",
    "direction_features"
]

for v in vars_to_check:
    print(v, "OK" if v in globals() else "MISSING")

daily_merge OK
news_features OK
earnings_features OK
analyst_features OK
extended_features OK
direction_features OK


In [91]:
final_features = daily_merge[
    [
        "date_et",
        "symbol",
        "ret",
        "ret_5d",
        "vol_5d",
        "next_ret",
        "next_label"
    ]
].copy()

In [92]:
final_features = final_features.merge(
    news_features[
        [
            "date_et",
            "symbol",
            "news_count",
            "news_count_ratio"
        ]
    ],
    on=["date_et", "symbol"],
    how="left"
)

In [93]:
earnings_tmp = earnings_features[
    [
        "date_et",
        "symbol",
        "earnings_recent_3d",
        "surprise_pct"
    ]
].copy()

earnings_tmp["abs_surprise_pct"] = (
    earnings_tmp["surprise_pct"].abs()
)

final_features = final_features.merge(
    earnings_tmp[
        [
            "date_et",
            "symbol",
            "earnings_recent_3d",
            "abs_surprise_pct"
        ]
    ],
    on=["date_et", "symbol"],
    how="left"
)

In [94]:
analyst_tmp = analyst_features[
    [
        "date_et",
        "symbol",
        "analyst_recent_3d",
        "target_lower",
        "target_change_pct"
    ]
].copy()

analyst_tmp["abs_target_change_pct"] = (
    analyst_tmp["target_change_pct"].abs()
)

final_features = final_features.merge(
    analyst_tmp[
        [
            "date_et",
            "symbol",
            "analyst_recent_3d",
            "target_lower",
            "abs_target_change_pct"
        ]
    ],
    on=["date_et", "symbol"],
    how="left"
)

In [95]:
direction_tmp = extended_features[
    [
        "date_et",
        "symbol",
        "premarket_move"
    ]
].copy()

direction_tmp = direction_tmp.merge(
    direction_features[
        [
            "date_et",
            "symbol",
            "premarket_range_position",
            "market_relative_premarket"
        ]
    ],
    on=["date_et", "symbol"],
    how="left"
)

final_features = final_features.merge(
    direction_tmp,
    on=["date_et", "symbol"],
    how="left"
)


In [96]:
print("shape:", final_features.shape)

print("\ncolumns:")
print(final_features.columns.tolist())

print("\nmissing values:")
display(final_features.isna().sum())

final_features.head()

shape: (36293, 17)

columns:
['date_et', 'symbol', 'ret', 'ret_5d', 'vol_5d', 'next_ret', 'next_label', 'news_count', 'news_count_ratio', 'earnings_recent_3d', 'abs_surprise_pct', 'analyst_recent_3d', 'target_lower', 'abs_target_change_pct', 'premarket_move', 'premarket_range_position', 'market_relative_premarket']

missing values:


date_et                          0
symbol                           0
ret                             50
ret_5d                         250
vol_5d                         250
next_ret                        50
next_label                      50
news_count                   10193
news_count_ratio             13795
earnings_recent_3d           10193
abs_surprise_pct             10292
analyst_recent_3d            10193
target_lower                 10250
abs_target_change_pct        12661
premarket_move               10266
premarket_range_position     10281
market_relative_premarket    10266
dtype: int64

,date_et,symbol,ret,ret_5d,vol_5d,next_ret,next_label,news_count,news_count_ratio,earnings_recent_3d,abs_surprise_pct,analyst_recent_3d,target_lower,abs_target_change_pct,premarket_move,premarket_range_position,market_relative_premarket
0,2023-10-20,ADI,NaN,NaN,NaN,-0.015855,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-10-23,ADI,-0.015855,NaN,NaN,0.006469,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-10-24,ADI,0.006469,NaN,NaN,-0.031286,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-10-25,ADI,-0.031286,NaN,NaN,0.006384,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-10-26,ADI,0.006384,NaN,NaN,-0.001306,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [97]:
official_start = day_meta["date_et"].min()
official_end = day_meta["date_et"].max()

final_features = final_features[
    (final_features["date_et"] >= official_start) &
    (final_features["date_et"] <= official_end)
].copy()

print(final_features.shape)
display(final_features.isna().sum())

(26100, 17)


date_et                         0
symbol                          0
ret                             0
ret_5d                          0
vol_5d                          0
next_ret                        0
next_label                      0
news_count                      0
news_count_ratio             3602
earnings_recent_3d              0
abs_surprise_pct               99
analyst_recent_3d               0
target_lower                   57
abs_target_change_pct        2468
premarket_move                 73
premarket_range_position       88
market_relative_premarket      73
dtype: int64

In [98]:
print("shape:", final_features.shape)
print("dates:", final_features["date_et"].nunique())
print("symbols:", final_features["symbol"].nunique())

shape: (26100, 17)
dates: 522
symbols: 50
